# User Activation Funnel Analysis

## Project Overview
Analyze onboarding and activation events to understand what user actions correlate with successful activation and where new users struggle most. An "activated" user is one who reaches a product's core value within a defined time window.

## Learning Objectives
- Define and measure user activation from event data
- Build activation funnels by cohort and segment
- Identify key actions that predict activation
- Analyze time-to-activation patterns and friction points

## Problem Statement
A SaaS product has a low activation rate — many users sign up but never experience the product's core value. The growth team wants to identify which onboarding actions predict activation and where the biggest friction points are.

## Why This Project Matters
Activation is the most critical metric in the user lifecycle. Users who don't activate within the first few sessions rarely return. Understanding the activation path and removing friction directly improves retention and revenue.

## Dataset Overview
Synthetic onboarding event data: ~5K new users, 7 onboarding actions tracked, activation status within 7 days. Includes signup source, device, and plan tier.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by SaaS product analytics
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))

ONBOARDING_ACTIONS = [
    'Complete Profile', 'Watch Tutorial', 'Connect Integration',
    'Invite Team Member', 'Create First Project', 'Upload Data', 'Run First Report'
]
ACTIVATION_ACTION = 'Run First Report'  # core value moment
ACTIVATION_WINDOW_DAYS = 7
print(f'Activation action: {ACTIVATION_ACTION}')
print(f'Activation window: {ACTIVATION_WINDOW_DAYS} days')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_users = 5000

sources = ['Organic', 'Paid', 'Referral', 'Social']
source_weights = [0.35, 0.25, 0.20, 0.20]
source_activation_mod = {'Organic': 1.0, 'Paid': 0.80, 'Referral': 1.15, 'Social': 0.75}

devices = ['Desktop', 'Mobile']
device_weights = [0.60, 0.40]
device_mod = {'Desktop': 1.0, 'Mobile': 0.80}

plans = ['Free', 'Trial', 'Paid']
plan_weights = [0.50, 0.30, 0.20]
plan_mod = {'Free': 0.75, 'Trial': 1.0, 'Paid': 1.20}

# Base probability of completing each action (conditional on signup)
action_base_probs = {
    'Complete Profile': 0.75,
    'Watch Tutorial': 0.55,
    'Connect Integration': 0.35,
    'Invite Team Member': 0.25,
    'Create First Project': 0.50,
    'Upload Data': 0.40,
    'Run First Report': 0.30,  # activation
}

signup_dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')

rows = []
for uid in range(n_users):
    source = np.random.choice(sources, p=source_weights)
    device = np.random.choice(devices, p=device_weights)
    plan = np.random.choice(plans, p=plan_weights)
    signup_date = np.random.choice(signup_dates)

    s_mod = source_activation_mod[source]
    d_mod = device_mod[device]
    p_mod = plan_mod[plan]
    combined_mod = s_mod * d_mod * p_mod

    actions_completed = []
    action_times = {}
    for action in ONBOARDING_ACTIONS:
        prob = min(action_base_probs[action] * combined_mod, 0.95)
        if np.random.random() < prob:
            actions_completed.append(action)
            # Time to complete (hours after signup)
            hours = max(0.1, np.random.exponential(scale=24 if action != 'Complete Profile' else 2))
            action_times[action] = round(hours, 1)

    activated = ACTIVATION_ACTION in actions_completed
    time_to_activate = action_times.get(ACTIVATION_ACTION, None)

    row = {
        'UserID': f'U{uid:05d}', 'SignupDate': signup_date,
        'Source': source, 'Device': device, 'Plan': plan,
        'ActionsCompleted': len(actions_completed),
        'Activated': activated,
        'TimeToActivateHours': time_to_activate,
    }
    for action in ONBOARDING_ACTIONS:
        row[f'Did_{action.replace(" ", "_")}'] = action in actions_completed
        row[f'Time_{action.replace(" ", "_")}'] = action_times.get(action, None)

    rows.append(row)

df = pd.DataFrame(rows)
df['SignupDate'] = pd.to_datetime(df['SignupDate'])
df['SignupMonth'] = df['SignupDate'].dt.to_period('M')

action_cols = [f'Did_{a.replace(" ", "_")}' for a in ONBOARDING_ACTIONS]
print(f'Dataset shape: {df.shape}')
print(f'Activation rate: {df["Activated"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values (key cols): {df[["UserID","Source","Device","Plan","Activated"]].isnull().sum().sum()}')
print(f'Unique users: {df["UserID"].nunique()}')
print(f'Date range: {df["SignupDate"].min().date()} to {df["SignupDate"].max().date()}')
print(f'Actions completed range: {df["ActionsCompleted"].min()} - {df["ActionsCompleted"].max()}')
print(f'Activated users: {df["Activated"].sum()} ({df["Activated"].mean():.1%})')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Activation rate by source
act_by_source = df.groupby('Source')['Activated'].mean().sort_values(ascending=True)
act_by_source.plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Activation Rate by Signup Source')
axes[0,0].set_xlabel('Activation Rate')
axes[0,0].set_xlim(0, 0.6)

# Activation rate by device
act_by_device = df.groupby('Device')['Activated'].mean()
act_by_device.plot.bar(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Activation Rate by Device')
axes[0,1].tick_params(axis='x', rotation=0)

# Actions completed distribution
df['ActionsCompleted'].hist(ax=axes[1,0], bins=range(0, 9), edgecolor='black', alpha=0.7)
axes[1,0].set_title('Onboarding Actions Completed per User')
axes[1,0].set_xlabel('# Actions')

# Activation rate by plan
act_by_plan = df.groupby('Plan')['Activated'].mean().reindex(['Free','Trial','Paid'])
act_by_plan.plot.bar(ax=axes[1,1], edgecolor='black', color='mediumpurple')
axes[1,1].set_title('Activation Rate by Plan')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Onboarding Action Completion Rates

In [ ]:
action_rates = df[action_cols].mean().sort_values(ascending=True)
action_rates.index = [c.replace('Did_', '').replace('_', ' ') for c in action_rates.index]

fig, ax = plt.subplots(figsize=(10, 6))
action_rates.plot.barh(ax=ax, edgecolor='black', color=plt.cm.Blues(np.linspace(0.3, 0.9, len(action_rates))))
ax.set_title('Onboarding Action Completion Rates')
ax.set_xlabel('Completion Rate')
ax.set_xlim(0, 1)
for i, (v, name) in enumerate(zip(action_rates.values, action_rates.index)):
    ax.text(v + 0.01, i, f'{v:.0%}', va='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'action_completion.png'), dpi=100, bbox_inches='tight')
plt.show()

## Actions Correlated with Activation

In [ ]:
activation_corr = pd.DataFrame({
    'Action': ONBOARDING_ACTIONS,
    'Completion_Rate': [df[f'Did_{a.replace(" ", "_")}'].mean() for a in ONBOARDING_ACTIONS],
    'Activation_Rate_If_Done': [
        df[df[f'Did_{a.replace(" ", "_")}']]['Activated'].mean() for a in ONBOARDING_ACTIONS
    ],
    'Activation_Rate_If_Not_Done': [
        df[~df[f'Did_{a.replace(" ", "_")}']]['Activated'].mean() for a in ONBOARDING_ACTIONS
    ],
})
activation_corr['Activation_Lift'] = (
    activation_corr['Activation_Rate_If_Done'] / activation_corr['Activation_Rate_If_Not_Done']
).round(2)
activation_corr = activation_corr.sort_values('Activation_Lift', ascending=False)
print('Action Impact on Activation:')
print(activation_corr.round(3).to_string(index=False))

## Activation Lift Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
lift_data = activation_corr.set_index('Action')
x = range(len(lift_data))
ax.bar([i-0.2 for i in x], lift_data['Activation_Rate_If_Done'], width=0.4,
       label='Did Action', edgecolor='black', color='#66bb6a')
ax.bar([i+0.2 for i in x], lift_data['Activation_Rate_If_Not_Done'], width=0.4,
       label='Did Not', edgecolor='black', color='#ef5350')
ax.set_xticks(x)
ax.set_xticklabels(lift_data.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Activation Rate')
ax.set_title('Activation Rate: Users Who Did vs Did Not Complete Each Action')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'activation_lift.png'), dpi=100, bbox_inches='tight')
plt.show()

## Time-to-Activation Analysis

In [ ]:
activated_users = df[df['Activated']].copy()
print(f'Activated users: {len(activated_users)}')
print(f'Time to activation (hours):')
print(activated_users['TimeToActivateHours'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 5))
activated_users['TimeToActivateHours'].hist(ax=ax, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(x=activated_users['TimeToActivateHours'].median(), color='red', linestyle='--',
           label=f'Median: {activated_users["TimeToActivateHours"].median():.1f}h')
ax.set_title('Time to Activation Distribution')
ax.set_xlabel('Hours After Signup')
ax.set_ylabel('Users')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'time_to_activation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Activation Rate by Cohort (Monthly)

In [ ]:
cohort_act = df.groupby('SignupMonth')['Activated'].mean()
fig, ax = plt.subplots(figsize=(12, 5))
cohort_act.plot(ax=ax, marker='o', color='green')
ax.set_title('Activation Rate by Signup Month')
ax.set_ylabel('Activation Rate')
ax.set_ylim(0, 0.5)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'activation_by_cohort.png'), dpi=100, bbox_inches='tight')
plt.show()

## Activation Funnel — Ordered by Typical Sequence

In [ ]:
ordered_actions = ['Complete Profile', 'Watch Tutorial', 'Create First Project',
                    'Upload Data', 'Connect Integration', 'Invite Team Member', 'Run First Report']
funnel_rates = [df[f'Did_{a.replace(" ", "_")}'].mean() for a in ordered_actions]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(ordered_actions[::-1], funnel_rates[::-1], edgecolor='black',
               color=plt.cm.Greens(np.linspace(0.3, 0.9, len(ordered_actions))))
ax.set_xlabel('Completion Rate')
ax.set_title('Onboarding Activation Funnel')
for bar, rate in zip(bars, funnel_rates[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{rate:.0%}', va='center')
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'activation_funnel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Struggle Points — Where New Users Get Stuck

In [ ]:
print('Activation Friction Points:')
print('=' * 60)
non_activated = df[~df['Activated']]
print(f'Non-activated users: {len(non_activated)} ({len(non_activated)/len(df):.0%})')
print(f'\nActions they DID complete:')
for action in ONBOARDING_ACTIONS:
    col = f'Did_{action.replace(" ", "_")}'
    rate = non_activated[col].mean()
    print(f'  {action}: {rate:.0%}')

print(f'\nActions completed by non-activated users:')
print(non_activated['ActionsCompleted'].value_counts().sort_index())

## Interpretation and Business Insight
- **Referral** users activate at the highest rate — they arrive with context and intent
- **Mobile** users activate significantly less than Desktop, likely due to complex onboarding flows
- **Create First Project** and **Upload Data** are the key "aha moment" precursors — users who do these are much more likely to reach activation
- **Connect Integration** and **Invite Team Member** have low completion but high activation lift — consider making them more prominent
- Non-activated users often complete 2-3 actions but stop before the core value action — the gap between exploration and commitment is the key friction zone
- **Paid** plan users activate more, suggesting commitment (sunk cost) helps motivation

## Limitations
- Synthetic data with pre-set probabilities; real activation patterns are more nuanced
- Actions are treated independently; in reality, the sequence matters
- No session-level data (how long users spend on each action)
- Activation defined by a single action — real products may have multi-dimensional activation criteria

## How to Improve This Project
- Use real product analytics data for more realistic patterns
- Add sequence analysis (what order do activated users complete actions?)
- Build a predictive model: given actions in first 24h, predict activation
- A/B test onboarding changes and measure activation impact
- Add multi-dimensional activation criteria (e.g., complete 3 of 5 key actions)

## Production Considerations
- Track activation rate as a company-level KPI on a weekly cadence
- Set up automated nudges when users stall at specific onboarding steps
- Personalize onboarding based on signup source and device
- Monitor activation rate by cohort to detect regressions from product changes

## Common Mistakes
- Defining activation too loosely (any login) or too strictly (requires all actions)
- Not segmenting activation by source and device (hides important differences)
- Treating onboarding as one-size-fits-all instead of personalizing by user intent
- Measuring activation once without tracking how it changes over time

## Mini Challenge / Exercises
1. What is the activation rate for users who complete 4+ actions vs 2 or fewer?
2. Build a "predicted activation score" based on which actions a user has completed.
3. Which single action, if its completion rate increased by 20%, would have the biggest impact on overall activation?
4. How does time-to-activation differ between Desktop and Mobile users?

## Final Summary / Key Takeaways
- Activation analysis is the foundation of product-led growth strategy
- A small number of key onboarding actions strongly predict activation
- Mobile onboarding needs simpler, more streamlined flows
- Referral users activate best — invest in referral programs
- The gap between "tried the product" and "experienced core value" is where most users are lost
- Personalized onboarding nudges at friction points can meaningfully improve activation rates